# 01 — Data ingest, validation & EDA

Run this notebook first. It inspects the three CSVs, logs cleaning decisions, and checks the shared feature pipeline.

**Kernel working directory:** repo root *or* `ml/notebooks` — the first code cell finds the project root.

The transformers themselves live in `ml/pipeline/features.py` so training **and** the later FastAPI service import the same object. Everything you *run* for EDA and modelling is in these notebooks.


In [ ]:
%matplotlib inline


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "ml" / "pipeline" / "features.py").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ml.config import (
    DATA_RAW,
    FIGURES_DIR,
    MODELS_DIR,
    RANDOM_STATE,
    REPORTS_DIR,
    ensure_dirs,
)

ensure_dirs()
print("Project root:", ROOT)


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ml.pipeline.features import DefaultRiskFeatures, LoanApprovalFeatures, LoanAmountFeatures, strip_loan_frame

sns.set_theme(style="whitegrid")

def find_csv(*names):
    for name in names:
        for folder in (DATA_RAW, ROOT):
            path = folder / name
            if path.exists():
                return path
    raise FileNotFoundError(names)

default_path = find_csv("credit_risk_dataset.csv", "credit_risk_dataset_1.csv")
loan_path = find_csv("loan_approval_dataset.csv")
german_path = find_csv("german_credit_data.csv")
print(default_path)
print(loan_path)
print(german_path)


## Load & validate


In [ ]:
default_df = pd.read_csv(default_path)
loan_df = strip_loan_frame(pd.read_csv(loan_path))
german_df = pd.read_csv(german_path)
if "Unnamed: 0" in german_df.columns:
    german_df = german_df.drop(columns=["Unnamed: 0"])

assert default_df.shape[1] == 12
assert "SeriousDlqin2yrs" in default_df.columns
assert "loan_status" in loan_df.columns and "loan_amount" in loan_df.columns

print("default", default_df.shape)
print("loan   ", loan_df.shape, list(loan_df.columns))
print("german ", german_df.shape, list(german_df.columns), "— no target column, EDA only")
default_df.head()


## Model A dataset — Give Me Some Credit style

~93% / 7% class split drives `scale_pos_weight`. `MonthlyIncome` is ~20% missing (median impute in the pipeline). Utilisation and debt ratio have extreme outliers (capped at 2.0 and 5.0). Late-payment codes **96 / 98** are sentinels, not real counts.


In [ ]:
print(default_df["SeriousDlqin2yrs"].value_counts(normalize=True))
print("\nMissing rates:")
print(default_df.isna().mean()[default_df.isna().mean() > 0])
print(default_df[["RevolvingUtilizationOfUnsecuredLines", "DebtRatio", "MonthlyIncome", "age"]].describe())

late_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]
for col in late_cols:
    print(col, "96/98 count:", int(((default_df[col] == 96) | (default_df[col] == 98)).sum()))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(x=default_df["SeriousDlqin2yrs"].map({0: "No default", 1: "Default"}), ax=axes[0])
axes[0].set_title("Target: SeriousDlqin2yrs")
sns.histplot(default_df["RevolvingUtilizationOfUnsecuredLines"].clip(upper=3), bins=40, ax=axes[1])
axes[1].set_title("Utilisation (clipped at 3 for the plot only)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_default_target.png", dpi=140)
plt.show()


## Model B / C dataset — loan approval

Headers had leading spaces (stripped on load). Mild imbalance (~62% Approved). `cibil_score` is the dominant driver — almost a decision rule. `loan_amount` is right-skewed (log1p for Model C). Negative residential assets are clipped at 0 in the pipeline.


In [ ]:
print(loan_df["loan_status"].value_counts())
print("CIBIL median by status:\n", loan_df.groupby("loan_status")["cibil_score"].median())
print("loan_amount skew:", round(loan_df["loan_amount"].skew(), 3))
print("asset mins:", loan_df[["residential_assets_value", "commercial_assets_value", "luxury_assets_value", "bank_asset_value"]].min().to_dict())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.countplot(data=loan_df, x="loan_status", ax=axes[0])
axes[0].set_title("loan_status")
sns.kdeplot(data=loan_df, x="cibil_score", hue="loan_status", common_norm=False, ax=axes[1])
axes[1].set_title("CIBIL by status")
sns.histplot(loan_df["loan_amount"], bins=40, ax=axes[2])
axes[2].set_title("Requested loan amount")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_loan_status.png", dpi=140)
fig.savefig(FIGURES_DIR / "eda_cibil_by_status.png", dpi=140)
fig.savefig(FIGURES_DIR / "eda_loan_amount.png", dpi=140)
plt.show()


## German credit — auxiliary only

This export has **no risk/target label**, so it is not used to train a model. Use it for feature ideas (purpose, housing, savings).


In [ ]:
print(german_df["Purpose"].value_counts())
german_df.head()


## Shared pipeline smoke test (same objects the models will use)


In [ ]:
feat_a = DefaultRiskFeatures().fit(default_df.drop(columns=["SeriousDlqin2yrs", "Id"], errors="ignore"))
out_a = feat_a.transform(default_df.head(3))
print("Model A engineered columns:", list(out_a.columns))
print("MonthlyIncome NaNs after transform:", int(out_a["MonthlyIncome"].isna().sum()))
print(out_a)

feat_b = LoanApprovalFeatures().fit(loan_df)
out_b = feat_b.transform(loan_df.head(3))
print("Model B columns:", list(out_b.columns))
print(out_b)

feat_c = LoanAmountFeatures().fit(loan_df)
out_c = feat_c.transform(loan_df.head(3))
assert "loan_amount" not in out_c.columns, "Model C must not leak the target"
print("Model C columns (no loan_amount):", list(out_c.columns))


## Cleaning log (for the report)

- **MonthlyIncome ~20% missing** — median impute at *fit* time, not a global constant.
- **Dependents missing** — fill 0.
- **Utilisation / DebtRatio** — cap 2.0 / 5.0; values like 50,000 are data errors.
- **96 / 98 late-payment codes** — replace with non-sentinel median.
- **age = 0** — impute median adult age.
- **Loan CSV spaces** — strip headers and categorical values immediately.
- **Negative residential assets** — clip at 0.
- **Model C** — train on Approved rows only; never use `loan_amount` as a feature; cap `min(requested, predicted)` in the API, not in training.
